# Rossmann Sales Forecasting - Feature Engineering

### The goal of this notebook is to prepare and create meaningful features that can help Machine Learning models predict daily sales across Rossmann stores.

### 1. Loading and Preparing Data

In [5]:
from pathlib import Path
import pandas as pd

In [6]:

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "data" / "processed").is_dir()
)

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

train_df = pd.read_csv(
    PROCESSED_PATH / "train_merged.csv",
    dtype={"state_holiday": "string"},
    parse_dates=["date"]
)

test_df = pd.read_csv(
    PROCESSED_PATH / "test_merged.csv",
    dtype={"state_holiday": "string"},
    parse_dates=["date"]
)

**Observation:** The cleaned and merged datasets are used to ensure consistency with the previous data preparation steps. Both datasets include store characteristics, while `sales` is available only in the training dataset.

### 2. Calendar-Based Feature Engineering

In [7]:
# Calendar-Based Feature Engineering

for df in [train_df, test_df]:

    df["date"] = pd.to_datetime(df["date"])

    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day_of_month"] = df["date"].dt.day
    df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

# Validate calendar features
calendar_features = [
    "date",
    "year",
    "month",
    "day_of_month",
    "week_of_year",
    "day_of_week"
]

display(train_df[calendar_features].head())

,date,year,month,day_of_month,week_of_year,day_of_week
0,2015-07-31,2015,7,31,31,5
1,2015-07-31,2015,7,31,31,5
2,2015-07-31,2015,7,31,31,5
3,2015-07-31,2015,7,31,31,5
4,2015-07-31,2015,7,31,31,5


**Observation:** Calendar features were extracted to capture temporal patterns in daily sales. The EDA identified seasonal variations, particularly higher sales in December, suggesting that time-related variables may improve forecasting performance.

### 3. Promotion-Based Feature Engineering

In [8]:
# Promotion-based feature engineering

month_names = {
    1: "Jan", 2: "Feb", 3: "Mar",
    4: "Apr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Aug", 9: "Sept",
    10: "Oct", 11: "Nov", 12: "Dec"
}

for df in [train_df, test_df]:

    # Convert Promo2 ISO year/week into a start date
    year = df["promo2_since_year"].astype("Int64").astype("string")
    week = df["promo2_since_week"].astype("Int64").astype("string")

    promo2_start = pd.to_datetime(
        year + "-" + week.str.zfill(2) + "-1",
        format="%G-%V-%u",
        errors="coerce"
    )

    # Calculate weeks since Promo2 started
    df["promo2_weeks_since_start"] = (
        ((df["date"] - promo2_start).dt.days // 7)
        .clip(lower=0)
        .fillna(0)
        .astype(int)
    )

    # Check whether the current month is a Promo2 month
    current_month = df["date"].dt.month.map(month_names)

    promo_month = pd.Series(False, index=df.index)

    for interval in df["promo_interval"].dropna().unique():
        mask = df["promo_interval"].eq(interval)

        promo_month.loc[mask] = current_month.loc[mask].isin(
            interval.split(",")
        )

    # Identify active Promo2 periods
    df["promo2_active"] = (
        (df["promo2"] == 1)
        & (df["date"] >= promo2_start)
        & promo_month
    ).astype(int)

In [9]:
# Validate promotion features

promo_features = [
    "store",
    "date",
    "promo",
    "promo2",
    "promo_interval",
    "promo2_active",
    "promo2_weeks_since_start"
]

display(train_df[promo_features].head(10))

# Check Promo2 activity
display(
    train_df.groupby("promo2")["promo2_active"]
    .value_counts()
    .to_frame("count")
)

,store,date,promo,promo2,promo_interval,promo2_active,promo2_weeks_since_start
0,1,2015-07-31,1,0,NaN,0,0
1,2,2015-07-31,1,1,"Jan,Apr,Jul,Oct",1,278
2,3,2015-07-31,1,1,"Jan,Apr,Jul,Oct",1,225
3,4,2015-07-31,1,0,NaN,0,0
4,5,2015-07-31,1,0,NaN,0,0
5,6,2015-07-31,1,0,NaN,0,0
6,7,2015-07-31,1,0,NaN,0,0
7,8,2015-07-31,1,0,NaN,0,0
8,9,2015-07-31,1,0,NaN,0,0
9,10,2015-07-31,1,0,NaN,0,0


count
promo2 promo2_active        
0      0              508031
1      0              357126
       1              152052

**Observation:** Two promotion-related features were created to capture whether Promo2 is active on a given date and how long the store has participated in the campaign.

These features provide additional temporal information that may help the model capture promotional patterns and improve sales predictions.

In [10]:
# Validate Promo2 features in both datasets
for name, df in [("Train", train_df), ("Test", test_df)]:
    invalid = (
        (df["promo2"] == 0) &
        (df["promo2_active"] == 1)
    ).sum()

    print(f"{name} - Invalid Promo2 records: {invalid}")


Train - Invalid Promo2 records: 0
Test - Invalid Promo2 records: 0


**Observation:** The new features distinguish stores participating in Promo2 from periods when the campaign is actually active. Validation confirmed that stores without Promo2 never have an active campaign, while participating stores may have inactive periods depending on the promotional calendar.

### 4. Competition-Based Feature Engineering

In [11]:
# Create competition-related features
for df in [train_df, test_df]:

    # Calculate months since competitor opening
    months_since = (
        (df["date"].dt.year - df["competition_open_since_year"]) * 12
        + (df["date"].dt.month - df["competition_open_since_month"])
    )

    # Avoid negative values for competitors not yet open
    df["competition_months_open"] = months_since.clip(lower=0)

    # Identify stores with known competition opening dates
    df["competition_start_known"] = (
        df["competition_open_since_year"].notna()
        & df["competition_open_since_month"].notna()
    ).astype(int)

In [12]:
# Validate competition features
competition_features = [
    "store",
    "date",
    "competition_distance",
    "competition_open_since_month",
    "competition_open_since_year",
    "competition_months_open",
    "competition_start_known"
]

display(train_df[competition_features].head(10))

display(
    train_df[
        ["competition_months_open", "competition_start_known"]
    ].describe()
)

,store,date,competition_distance,competition_open_since_month,competition_open_since_year,competition_months_open,competition_start_known
0,1,2015-07-31,1270.0,9.0,2008.0,82.0,1
1,2,2015-07-31,570.0,11.0,2007.0,92.0,1
2,3,2015-07-31,14130.0,12.0,2006.0,103.0,1
3,4,2015-07-31,620.0,9.0,2009.0,70.0,1
4,5,2015-07-31,29910.0,4.0,2015.0,3.0,1
5,6,2015-07-31,310.0,12.0,2013.0,19.0,1
6,7,2015-07-31,24000.0,4.0,2013.0,27.0,1
7,8,2015-07-31,7520.0,10.0,2014.0,9.0,1
8,9,2015-07-31,2030.0,8.0,2000.0,179.0,1
9,10,2015-07-31,3160.0,9.0,2009.0,70.0,1


,competition_months_open,competition_start_known
count,693861.000000,1.017209e+06
mean,61.631076,6.821224e-01
std,71.079571,4.656519e-01
min,0.000000,0.000000e+00
25%,14.000000,0.000000e+00
50%,51.000000,1.000000e+00
75%,94.000000,1.000000e+00
max,1386.000000,1.000000e+00


In [13]:
# Investigate unusually old competition opening dates
extreme_competition = train_df.loc[
    train_df["competition_months_open"] > 600,
    [
        "store",
        "competition_open_since_month",
        "competition_open_since_year",
        "competition_months_open"
    ]
]

display(
    extreme_competition
    .sort_values("competition_months_open", ascending=False)
    .drop_duplicates(subset=["store"])
    .head(10)
)

,store,competition_open_since_month,competition_open_since_year,competition_months_open
16424,815,1.0,1900.0,1386.0
145,146,10.0,1961.0,645.0


**Observation:** Competition opening dates are available for approximately 68.2% of training observations. The new features capture how long a competitor has been operating and whether its opening date is known.

Two stores have unusually old competition opening dates (1900 and 1961). These values were retained, as there is insufficient evidence to classify them as errors. Their potential impact will be assessed during model training and evaluation.

### 5. Categorical Feature Preparation

In [14]:
# Identify categorical features
categorical_features = [
    "store_type",
    "assortment",
    "state_holiday",
    "promo_interval"
]

# Compare unique categories
category_summary = pd.DataFrame({
    "Train Categories": train_df[categorical_features].nunique(),
    "Test Categories": test_df[categorical_features].nunique(),
    "Train Missing": train_df[categorical_features].isna().sum(),
    "Test Missing": test_df[categorical_features].isna().sum()
})

display(category_summary)

,Train Categories,Test Categories,Train Missing,Test Missing
store_type,4,4,0,0
assortment,3,3,0,0
state_holiday,4,2,0,0
promo_interval,3,3,508031,17232


In [15]:
# Standardize categorical features
for df in [train_df, test_df]:

    # Ensure consistent string types
    for col in categorical_features:
        df[col] = df[col].astype("string")

    # Represent stores without Promo2
    df["promo_interval"] = (
        df["promo_interval"].fillna("NoPromo2")
    )

# Validate categorical features
display(
    train_df[categorical_features].head(10)
)

display(
    test_df[categorical_features].head(10)
)

,store_type,assortment,state_holiday,promo_interval
0,c,a,0,NoPromo2
1,a,a,0,"Jan,Apr,Jul,Oct"
2,a,a,0,"Jan,Apr,Jul,Oct"
3,c,c,0,NoPromo2
4,a,a,0,NoPromo2
5,a,a,0,NoPromo2
6,a,c,0,NoPromo2
7,a,a,0,NoPromo2
8,a,c,0,NoPromo2
9,a,a,0,NoPromo2


,store_type,assortment,state_holiday,promo_interval
0,c,a,0,NoPromo2
1,a,a,0,"Jan,Apr,Jul,Oct"
2,a,c,0,NoPromo2
3,a,a,0,NoPromo2
4,a,c,0,NoPromo2
5,a,a,0,NoPromo2
6,a,c,0,"Jan,Apr,Jul,Oct"
7,a,c,0,"Jan,Apr,Jul,Oct"
8,d,a,0,"Feb,May,Aug,Nov"
9,a,a,0,"Jan,Apr,Jul,Oct"


**Observation:** Categorical features were standardized to ensure consistent data types across the training and test datasets. Missing `promo_interval` values were assigned a separate category (`NoPromo2`), as they represent stores that do not participate in the promotion.

Categorical encoding will be fitted on the training data after the chronological split to prevent data leakage.

### 6. Final Feature Validation

In [16]:
# Validate engineered datasets
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nMissing values in new features:")

new_features = [
    "year",
    "month",
    "day_of_month",
    "week_of_year",
    "promo2_active",
    "promo2_weeks_since_start",
    "competition_months_open",
    "competition_start_known"
]

display(train_df[new_features].isna().sum())
display(test_df[new_features].isna().sum())

Train shape: (1017209, 26)
Test shape: (41088, 25)

Missing values in new features:


year                             0
month                            0
day_of_month                     0
week_of_year                     0
promo2_active                    0
promo2_weeks_since_start         0
competition_months_open     323348
competition_start_known          0
dtype: int64

year                            0
month                           0
day_of_month                    0
week_of_year                    0
promo2_active                   0
promo2_weeks_since_start        0
competition_months_open     15216
competition_start_known         0
dtype: int64

### 7. Save Feature-Engineered Datasets

**Observation:** The engineered datasets are saved separately from the merged inputs so subsequent modelling steps can use the same reproducible features without repeating the transformations.

In [17]:
# Save feature-engineered datasets without overwriting the merged inputs
train_df.to_csv(
    PROCESSED_PATH / "train_features.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_PATH / "test_features.csv",
    index=False
)

print("Feature-engineered datasets saved successfully.")

Feature-engineered datasets saved successfully.
